In [16]:
import pandas as pd 
import numpy as np

# Load the original dataset
df = pd.read_csv('../data/Teen_Mental_Health_Dataset.csv')

# Check the shape of the dataset
print(f"Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns.")
print("Detected columns:")
print(df.columns.to_list())

# Display the first 3 rows to verify everything loaded correctly
df.head(3)

Dataset shape: 1200 rows and 13 columns.
Detected columns:
['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0


In [17]:
# ==========================================
# 2. MISSING VALUES HANDLING
# ==========================================
# Check for null values across columns
print("Missing values per column before cleansing:")
print(df.isnull().sum())
print("-" * 40)

# Imputation strategy:
# - For essential numerical features (like age or metrics): fill with median to avoid outlier distortion.
# - For categorical features (like gender or platform): fill with mode (most frequent value).
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

Missing values per column before cleansing:
age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64
----------------------------------------


C:\Users\52444\AppData\Local\Temp\ipykernel_22744\2595451106.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


In [18]:
# ==========================================
# 3. TEXT STANDARDISATION (CATEGORICAL DATA)
# ==========================================
# Remove leading/trailing whitespaces and enforce uniform case formatting
df['gender'] = df['gender'].astype(str).str.strip().str.capitalize()
df['platform_usage'] = df['platform_usage'].astype(str).str.strip().str.title()
df['depression_label'] = df['depression_label'].astype(str).str.strip().str.capitalize()

# Standardize academic performance or text-based levels if they contain trailing spaces
if df['academic_performance'].dtype == 'object':
    df['academic_performance'] = df['academic_performance'].astype(str).str.strip().str.lower()

In [19]:
# ==========================================
# 4. BUSINESS RULE VALIDATION & OUTLIERS
# ==========================================
# Rule 1: Time metrics (sleep, social media, screen time) cannot be negative or exceed 24 hours in a single day.
# If anomalies exist, cap them using logical thresholds.
df['sleep_hours'] = df['sleep_hours'].clip(lower=0, upper=24)
df['daily_social_media_hours'] = df['daily_social_media_hours'].clip(lower=0, upper=24)
df['screen_time_before_sleep'] = df['screen_time_before_sleep'].clip(lower=0, upper=24)

# Rule 2: Ensure the age column contains integers
df['age'] = df['age'].astype(int)

In [20]:
# ==========================================
# 5. DATA TYPE CONVERSION (CASTING)
# ==========================================
# Cast psychological scale metrics (stress, anxiety, addiction) into clean integer formats
scale_cols = ['stress_level', 'anxiety_level', 'addiction_level']
for col in scale_cols:
    df[col] = df[col].astype(int)

In [21]:
# ==========================================
# 6. EXPORT STANDARDISED DATASET
# ==========================================
print(f"\n--- Final Standardised Dataset Preview ---")
print(df.head())

# Save the cleansed data to be consumed by the dimensional model script
df.to_csv('../data/social_media_mental_health_clean.csv', index=False)
print("\nSuccess! Cleansed dataset saved as 'social_media_mental_health_clean.csv'.")


--- Final Standardised Dataset Preview ---
   age  gender  daily_social_media_hours platform_usage  sleep_hours  \
0   14    Male                       7.9      Instagram          7.4   
1   19  Female                       1.9         Tiktok          8.0   
2   17  Female                       1.3      Instagram          7.6   
3   15    Male                       7.4         Tiktok          6.9   
4   15  Female                       4.7           Both          4.9   

   screen_time_before_sleep  academic_performance  physical_activity  \
0                       2.9                  3.01                1.5   
1                       2.9                  3.22                0.8   
2                       0.5                  3.92                0.0   
3                       1.6                  3.48                0.8   
4                       3.0                  2.37                1.4   

  social_interaction_level  stress_level  anxiety_level  addiction_level  \
0             